In [ ]:
from queue import PriorityQueue

class Point:
    def __init__(self, location, previous=None):
        self.location = location
        self.previous = previous
        self.cost_from_start = 0
        self.heuristic_value = 0
        self.total_estimate = 0
    
    def __lt__(self, other):
        return self.total_estimate < other.total_estimate

def compute_heuristic(current_point, target_point):
    return abs(current_point[0] - target_point[0]) + abs(current_point[1] - target_point[1])

def greedy_search_multiple_targets(environment, origin, targets):
    row_count, col_count = len(environment), len(environment[0])
    remaining_targets = targets.copy()
    current_location = origin
    complete_route = [origin]
    overall_cost = 0
    
    while remaining_targets:
        closest_target = None
        smallest_distance = float('inf')
        
        for target in remaining_targets:
            distance = compute_heuristic(current_location, target)
            if distance < smallest_distance:
                smallest_distance = distance
                closest_target = target
        
        route_to_target = greedy_search_single_target(environment, current_location, closest_target)
        
        if route_to_target:
            if len(route_to_target) > 1:
                complete_route.extend(route_to_target[1:])
                overall_cost += len(route_to_target) - 1
            
            current_location = closest_target
            remaining_targets.remove(closest_target)
        else:
            return None, float('inf')
    
    return complete_route, overall_cost

def greedy_search_single_target(environment, origin, target):
    row_count, col_count = len(environment), len(environment[0])
    start_point = Point(origin)
    destination_point = Point(target)
    
    priority_queue = PriorityQueue()
    priority_queue.put(start_point)
    explored = set()
    
    while not priority_queue.empty():
        current_point = priority_queue.get()
        current_coords = current_point.location
        
        if current_coords == target:
            route = []
            while current_point:
                route.append(current_point.location)
                current_point = current_point.previous
            return route[::-1]
        
        explored.add(current_coords)
        
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_coords = (current_coords[0] + dx, current_coords[1] + dy)
            
            if (0 <= new_coords[0] < row_count and 
                0 <= new_coords[1] < col_count and 
                environment[new_coords[0]][new_coords[1]] == 0 and 
                new_coords not in explored):
                
                new_point = Point(new_coords, current_point)
                new_point.cost_from_start = current_point.cost_from_start + 1
                new_point.heuristic_value = compute_heuristic(new_coords, target)
                new_point.total_estimate = new_point.heuristic_value
                
                priority_queue.put(new_point)
                explored.add(new_coords)
    
    return None

environment = [
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 0, 0, 0, 1, 0]
]

origin = (0, 0)
targets = [(2, 7), (5, 7), (7, 3), (4, 4)]

print("Multi-Goal Path Finding")
print(f"Starting Point: {origin}")
print(f"Destination Points: {targets}")

route, cost = greedy_search_multiple_targets(environment, origin, targets)

if route:
    print(f"Route covering all destinations: {route}")
    print(f"Total moves required: {cost}")
else:
    print("Unable to find a route to all destinations!")
print()

PROBLEM #1: Advanced Multi-Goal Path Finding
Starting Point: (0, 0)
Destination Points: [(2, 7), (5, 7), (7, 3), (4, 4)]
Route covering all destinations: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (5, 7), (4, 7), (3, 7), (2, 7), (3, 7), (4, 7), (5, 7), (5, 6), (5, 5), (6, 5), (7, 5), (7, 4), (7, 3)]
Total moves required: 24

